In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import pickle
from scipy.special import expit
rng = np.random.default_rng(0)

#### helpers

Based on code from `https://github.com/jacobmchen/proximal_w_text`. The file `preprocessing_util.py` can be downloaded there.

In [ ]:
from preprocessing_util import *

#### read data

Source data can be download on `https://physionet.org/content/mimiciii/1.4/`.

In [ ]:
# read notes data and sort
data_dir = './source_data/mimic-iii-clinical-database-1.4/'
notes = pd.read_csv(data_dir + 'NOTEEVENTS.csv' , low_memory=False).sort_values(by='CHARTDATE')

In [ ]:
# read diagnosis data
diagnoses_icd = pd.read_csv(data_dir + 'DIAGNOSES_ICD.csv')
diagnoses_icd_key = pd.read_csv(data_dir + 'D_ICD_DIAGNOSES.csv')

In [ ]:
# read patient data
patients = pd.read_csv(data_dir + 'PATIENTS.csv')
patients['gender'] = patients['GENDER'].map({'F': 0, 'M': 1})

#### process features

notes

In [ ]:
# drop missing IDS
notes = notes.dropna(subset=['HADM_ID'])

In [ ]:
# earliest record per patient
unique_subjects = set(notes['SUBJECT_ID'])
first_hadm_ids = []

# loop over subjects
for subject in unique_subjects:
    temp = notes[notes['SUBJECT_ID'] == subject]
    first_hadm_ids.append(temp.iloc[0]['HADM_ID'])

# select
notes_subset = notes[notes['HADM_ID'].isin(first_hadm_ids)]

In [ ]:
# remove discharge summaries
output_A = notes_subset[notes_subset['CATEGORY'] != 'Discharge summary']

diagnosis

In [ ]:
# clean
diagnoses_icd_key = diagnoses_icd_key.drop(columns=['LONG_TITLE', 'ROW_ID'])
output_B = diagnoses_icd.merge(diagnoses_icd_key, how='left', on='ICD9_CODE')
output_B = output_B.dropna(subset=['SHORT_TITLE'])

In [ ]:
# top 10 most frequent diagnosis
diagnosis_names = ['Hypertension NOS','Crnry athrscl natve vssl','Atrial fibrillation','CHF NOS','DMII wo cmp nt st uncntr','Hyperlipidemia NEC/NOS',
                   'Acute kidney failure NOS','Need prphyl vc vrl hepat','NB obsrv suspct infect','Acute respiratry failure']

In [ ]:
# retain top diagnosis per hospital admission and subject ID
diagnoses_df = output_A[['HADM_ID', 'SUBJECT_ID']].drop_duplicates()

# loop over top diagnosis and admissions
for diagnosis in diagnosis_names:
    l = []
    for idx, hadm_id in enumerate(diagnoses_df["HADM_ID"]):

        # check if admission is linked with diagnosis
        subset = output_B[output_B['HADM_ID'] == hadm_id]
        subset = subset[subset['SHORT_TITLE'] == diagnosis]
        if len(subset) >= 1:
            l.append(1)
        else:
            l.append(0)
    diagnoses_df[diagnosis] = l

patients

In [ ]:
# add gender and age to the clinical notes
output_A = output_A.merge(patients.set_index('SUBJECT_ID')[['gender','DOB']], left_on='SUBJECT_ID', right_index=True, how='left')
output_A['age'] = calculate_age(output_A['CHARTDATE'], output_A['DOB'])

# filter extremes
output_A = output_A[output_A['age'] > 18]
output_A = output_A[output_A['age'] < 100]

# add diagnosis
features = output_A.merge(diagnoses_df, on=['HADM_ID', 'SUBJECT_ID'], how='left')

#### generate treatments and outcomes

In [ ]:
# select relevant columns and rename
features = features[['Crnry athrscl natve vssl', 'Atrial fibrillation', 'Hypertension NOS', 'CHF NOS', 'age', 'gender']]
features.columns = ['CORONARY_ATHERO','ATRIAL_FIBRI','HYPERTENSION','CONGESTIVE_HF', 'AGE', 'GENDER']

In [ ]:
# copy df for data generation
df = features.copy()
df['AGE_c'] = df['AGE'] - 65

In [ ]:
df['M0'] = (
    5.5
    + 0.10 * df['AGE_c']                  
    + 0.60 * df['GENDER']                 
    + 0.70 * df['HYPERTENSION']           
    + 1.00 * df['CORONARY_ATHERO']        
    + 0.85 * df['ATRIAL_FIBRI']           
    + 1.20 * df['CONGESTIVE_HF']          
    + 0.020 * np.maximum(df['AGE'] - 75, 0) ** 2
    + 0.50 * df['CORONARY_ATHERO'] * df['CONGESTIVE_HF']     
    + 0.35 * df['ATRIAL_FIBRI'] * df['CONGESTIVE_HF']        
    + 0.25 * df['HYPERTENSION'] * df['CORONARY_ATHERO']      
    + 0.20 * df['GENDER'] * df['CORONARY_ATHERO']            
    + 0.015 * df['AGE_c'] * df['ATRIAL_FIBRI'])

In [ ]:
df['logit_e'] = (
    -0.8
    + 0.08 * df['AGE_c']
    + 0.65 * df['GENDER']
    + 0.50 * df['HYPERTENSION']
    + 0.60 * df['CORONARY_ATHERO']
    + 0.40 * df['ATRIAL_FIBRI']
    + 0.70 * df['CONGESTIVE_HF']
    + 0.25 * df['GENDER'] * (df['AGE'] > 75).astype(float)
    + 0.25 * df['CORONARY_ATHERO'] * df['CONGESTIVE_HF'])

df['e'] = expit(0.9 * df['logit_e'])
df['T'] = np.random.binomial(1, df['e'])

In [ ]:
df['cate'] = (
    -1.0
    - 0.35 * df['HYPERTENSION']
    - 0.50 * df['CORONARY_ATHERO']
    + 0.20 * df['ATRIAL_FIBRI']
    + 0.45 * df['CONGESTIVE_HF']
    + 0.015 * np.maximum(df['AGE'] - 75, 0)
    + 0.05 * df['GENDER']
    + 0.20 * df['ATRIAL_FIBRI'] * df['CONGESTIVE_HF']
    - 0.35 * df['HYPERTENSION'] * df['CORONARY_ATHERO'])

df['M1'] = df['M0'] + df['cate']

In [ ]:
noise_sd = 1.0
df['Y0'] = df['M0'] + np.random.normal(0, noise_sd, size=len(df))
df['Y1'] = df['M1'] + np.random.normal(0, noise_sd, size=len(df))
df['Y']  = df['T'] * df['Y1'] + (1 - df['T']) * df['Y0']

In [ ]:
# store
df.to_csv('./datasets/mimic_syn.csv')